# Neural Topic Modeling with BERTopic
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/05_NLP_Embeddings/bertopic_topic_modeling.ipynb)

BERTopic replaces word counting with embeddings: documents are embedded with sentence-transformers, clustered (HDBSCAN), and each cluster's theme is extracted with class-based TF-IDF. It catches paraphrases that LDA's bag-of-words misses.

Same 20 Newsgroups slice as `topic_modeling_lda.ipynb` so you can compare directly. Free Colab; first run downloads an ~80 MB embedding model.

In [ ]:
!pip install -q bertopic sentence-transformers scikit-learn

## 1. Data

In [ ]:
from sklearn.datasets import fetch_20newsgroups

cats = ["sci.space", "rec.sport.baseball", "talk.politics.mideast", "comp.graphics"]
subset = fetch_20newsgroups(subset="train", categories=cats,
                            remove=("headers", "footers", "quotes"))
docs = [d[:1500] for d in subset.data][:900]      # trim long posts, cap corpus
labels = subset.target[:900]
print(len(docs), "documents")

## 2. Fit BERTopic

In [ ]:
from bertopic import BERTopic

model = BERTopic(embedding_model="all-MiniLM-L6-v2",
                 min_topic_size=15,
                 verbose=False)
topics, probs = model.fit_transform(docs)
print(f"discovered {len(model.get_topic_info()) - 1} topics (topic -1 = outliers)")

## 3. Inspect discovered themes

In [ ]:
freq = model.get_topic_info()
freq.head(8)[["Topic", "Count", "Name"]]

In [ ]:
for t in freq.Topic.head(4):
    if t == -1:
        continue
    print(f"topic {t}:", ", ".join(w for w, _ in model.get_topic(t)[:8]))
    print()

Compare with LDA output: BERTopic groups 'shuttle launch nasa orbit' style themes even when posts share zero exact keywords.

## 4. Do topics align with the 4 newsgroups?

In [ ]:
import pandas as pd
valid = [(t, l) for t, l in zip(topics, labels) if t != -1]
ct = pd.crosstab(pd.Series([t for t, _ in valid], name="topic"),
                 pd.Series([subset.target_names[l].split(".")[-1] for _, l in valid],
                           name="newsgroup"))
ct

## 5. Search topics semantically

In [ ]:
similar = model.find_topics("rocket launch satellite")
for t in similar[0][:3]:
    print(t, "->", ", ".join(w for w, _ in model.get_topic(t)[:6]))

## LDA vs BERTopic
| | LDA | BERTopic |
|---|---|---|
| representation | word counts | transformer embeddings |
| paraphrase awareness | none | strong |
| outliers | forced into topics | HDBSCAN marks -1 |
| speed on big corpora | fast | heavier (embedding pass) |
| interpretability tools | top words | barcharts/hierarchy/heatmap built-in |

Rule of thumb: short/clean corpora -> either; messy real-world text -> BERTopic.